# Analysis of generated initial conditions

TODO

In [ ]:
import numpy as np
from pathlib import Path

import h5py
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from mpl_toolkits.axes_grid1 import make_axes_locatable

import sys
root = Path.cwd().parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from stepsic.parameters import CosmoParameters
from stepsic.data import CosmoData

In [ ]:
import logging
log = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO)

In [ ]:
# Facecolor values from S. Conradi @S_Conradi/@profConradi
custom_settings = {
    'figure.facecolor': '#ffffff',
    # 'figure.facecolor': '#f4f0e8',
    'axes.facecolor': '#ffffff',
    # 'axes.facecolor': '#f4f0e8',
    'axes.edgecolor': '0.3',
    'axes.linewidth' : '0.5',
    'axes.grid': False,
    'grid.color': '0.7',
    'grid.linestyle': ':',
    'grid.alpha': 0.6,
    'xtick.bottom': True,
    'xtick.top': True,
    'ytick.left': True,
    'ytick.right': True,
}
for t in ['xtick', 'ytick']:
    custom_settings[f'{t}.direction'] = 'in'
    custom_settings[f'{t}.color'] = '0.3'
    for m in ['major', 'minor']:
        custom_settings[f'{t}.{m}.width'] = 0.5
        custom_settings[f'{t}.{m}.size'] = 6 if m == 'major' else 3
sns.set_theme(palette=sns.color_palette('deep', as_cmap=False),
              rc=custom_settings)
plt.rcParams['text.usetex'] = False

In [ ]:
def inspect_glass_file(filename):
    with h5py.File(filename, 'r') as f:
        print('File structure:')
        def print_structure(name, obj):
            print(f'  {name}: {type(obj).__name__}')
            if hasattr(obj, 'shape'):
                print(f'    Shape: {obj.shape}')
            if hasattr(obj, 'dtype'):
                print(f'    Dtype: {obj.dtype}')
        
        f.visititems(print_structure)
        
        # Check specific attributes
        if 'Header' in f and 'BoxSize' in f['Header'].attrs:
            print(f"BoxSize: {f['Header'].attrs['BoxSize']}")
        
        if 'PartType1/Coordinates' in f:
            coords = f['PartType1/Coordinates'][:]
            print(f'Coordinates shape: {coords.shape}')
            print(f'First few particles:\n{coords[:3]}')

In [ ]:
params = CosmoParameters(path=Path('config.toml')).get_parameters()

In [ ]:
ic_orig = CosmoData.load_snapshot(Path(params['INPUT_GLASS']))
ic_orig.to_internal_units(params)
ic_orig.rescale_snapshot_mass(params)
ic_orig.periodic_shift(params)

ic = CosmoData.load_snapshot(Path('../output/stepsic_L1000_1000_500_R500_D75_z63.hdf5'))
ic.to_internal_units(params)
ic.rescale_snapshot_mass(params)
ic.periodic_shift(params)

In [ ]:
nmesh = params['NMESH']
Lbox = params['LBOX']

In [ ]:
nr, nc = 1, 3
fig, axes = plt.subplots(nr, nc, figsize=(nc*5, nr*4), dpi=120)

labels = ['x', 'y', 'z']
for i, ax in enumerate(axes.flat[:3]):
    ax.set_aspect(1)
    idx = [k for k in range(3) if k != i]

    ax.scatter(ic.pos.T[idx[0]], ic.pos.T[idx[1]], s=1**2, c='k', alpha=0.5)
    ax.set_xlim(-Lbox[idx[0]]/2, Lbox[idx[0]]/2)
    ax.set_ylim(-Lbox[idx[1]]/2, Lbox[idx[1]]/2)
    ax.set_xlabel(f'{labels[idx[0]]} [Mpc/h]', fontsize=10)
    ax.set_ylabel(f'{labels[idx[1]]} [Mpc/h]', fontsize=10)
plt.show()

In [ ]:
def histogram(x, bins=50):
    '''TODO'''
    hist, edge = np.histogram(x, bins=bins)
    bin_c = (edge[:-1] + edge[1:]) / 2
    bin_w = np.diff(edge)  # Width of each bin
    return bin_c, hist, bin_w

In [ ]:
nr, nc = 1, 2
fig, axes = plt.subplots(nr, nc, figsize=(nc*5, nr*4), dpi=120)
axes = iter(axes.flat)

ax = next(axes)
pos = ic.pos.copy()
pos[:, 2] += Lbox[2]/2
dis = np.linalg.norm((pos - ic_orig.pos)*1000, axis=1)
c, h, w = histogram(dis, bins=100)
ax.bar(c, h, w, lw=0.5, color='tab:blue', alpha=0.5)
ax.set_yscale('log')
ax.set_xlabel('Displacement [~kpc]', fontsize=10)
ax.set_title(f'StepsIC 2-LPT displacements [z={params["REDSHIFT"]}]', loc='left', fontsize=10)

ax = next(axes)
vel = np.linalg.norm((ic.vel - ic_orig.vel)/np.sqrt(params['SCALE']), axis=1)
c, h, w = histogram(vel, bins=100)
ax.bar(c, h, w, lw=0.5, color='tab:blue', alpha=0.5)
ax.set_xlabel('Velocity [km/s]', fontsize=10)
ax.set_title(f'StepsIC 2-LPT velocities [z={params["REDSHIFT"]}]', loc='left', fontsize=10)

plt.show()

#### Calculate $P(k)$ from simulation

In [ ]:
from nbodykit.lab import ArrayCatalog, ArrayMesh, FieldMesh, FFTPower

In [ ]:
def pk_particle(
        x, nvox, Lbox, *, resampler='tsc', interlaced=True):
    '''TODO'''
    catalog = ArrayCatalog({'x': x}, BoxSize=Lbox)
    mesh = catalog.to_mesh(
        Nmesh=nvox,
        resampler=resampler,
        interlaced=interlaced,  # Cancel the Alias effect in Fourier modes
        compensated=True,  # Corrects for MAS smoothing
        position='x'
    )
    kmin = 2.0 * np.pi / np.max(Lbox) # Fundamental mode
    kmax = np.sqrt(3) * np.pi * nvox[0] / Lbox[0]  # Nyquist frequency
    r = FFTPower(mesh, mode='1d', dk=kmin, kmin=kmin, kmax=kmax)  # TODO: PR for `kmax`
    return r.power

In [ ]:
def pk_field(field, Lbox):
    '''TODO'''
    nvox = field.shape
    mesh = ArrayMesh(field, BoxSize=Lbox)
    kmin = 2.0 * np.pi / np.max(Lbox) # Fundamental mode
    kmax = np.sqrt(3) * np.pi * nvox[0] / Lbox[0]  # Nyquist frequency
    r = FFTPower(mesh, mode='1d', dk=kmin/2, kmin=kmin, kmax=kmax)  # TODO: PR for `kmax`
    return r.power

### 1-LPT

In [ ]:
nr, nc = 1, 2
fig, axes = plt.subplots(nr, nc, figsize=(nc*5, nr*4), dpi=120)
axes = iter(axes.flat)

ax = next(axes)
ax.set_box_aspect(1)
ax.loglog(kh, pk, color='k', lw=1.5, label='CAMB P(k)', alpha=0.5)
pk_pt = pk_particle(ic.pos, nvox, Lbox)
ax.loglog(pk_pt['k'], pk_pt['power'].real, color='tab:red', lw=1.5, label='Particle P(k)')
ax.set_xlabel('$k\,[h/\mathrm{Mpc}]$', fontsize=10)
ax.set_ylabel(r'$P(k)\,[h^{-3}\mathrm{Mpc}^3]$', fontsize=10)
ax.set_title('Matter power spectrum of perturbed positions', loc='left', fontsize=10)
ax.legend(loc='upper right', fontsize=10, frameon=False)

ax = next(axes)
ax.set_box_aspect(1)
ax.loglog(kh, pk, color='k', lw=1.5, label='CAMB P(k)', alpha=0.5)
pk_delta = pk_field(np.fft.irfftn(delta_k), Lbox)
ax.loglog(pk_delta['k'], pk_delta['power'].real, color='tab:green', lw=1.5, label='Field P(k)')
ax.set_xlabel('$k\,[h/\mathrm{Mpc}]$', fontsize=10)
ax.set_ylabel(r'$P(k)\,[h^{-3}\mathrm{Mpc}^3]$', fontsize=10)
ax.set_title(r'Matter power spectrum of $\delta(x)$', loc='left', fontsize=10)
ax.legend(loc='upper right', fontsize=10, frameon=False)

plt.savefig(f'output/pk_grid_N{nmesh}_L{np.max(Lbox)}_1lpt.png', bbox_inches='tight', dpi=300)
plt.show()

### 2-LPT

In [ ]:
nr, nc = 1, 2
fig, axes = plt.subplots(nr, nc, figsize=(nc*5, nr*4), dpi=120)
axes = iter(axes.flat)

ax = next(axes)
ax.set_box_aspect(1)
ax.loglog(kh, pk, color='k', lw=1.5, label='CAMB P(k)', alpha=0.5)
pk_pt = pk_particle(ic.pos, nvox, Lbox)
ax.loglog(pk_pt['k'], pk_pt['power'].real, color='tab:red', lw=1.5, label='Particle P(k)')
ax.set_xlabel('$k\,[h/\mathrm{Mpc}]$', fontsize=10)
ax.set_ylabel(r'$P(k)\,[h^{-3}\mathrm{Mpc}^3]$', fontsize=10)
ax.set_title('Matter power spectrum (2-LPT)', loc='left', fontsize=10)
ax.legend(loc='upper right', fontsize=10, frameon=False)

ax = next(axes)
ax.set_box_aspect(1)
ax.loglog(kh, pk, color='k', lw=1.5, label='CAMB P(k)', alpha=0.5)
pk_delta = pk_field(np.fft.irfftn(delta_k), Lbox)
ax.loglog(pk_delta['k'], pk_delta['power'].real, color='tab:green', lw=1.5, label='Particle P(k)')
ax.set_xlabel('$k\,[h/\mathrm{Mpc}]$', fontsize=10)
ax.set_title(r'Matter power spectrum of $\delta(x)$', loc='left', fontsize=10)
ax.legend(loc='upper right', fontsize=10, frameon=False)

plt.savefig(f'output/pk_grid_N{nmesh}_L{np.max(Lbox)}_2lpt.png', bbox_inches='tight', dpi=300)
plt.show()

### 1-LPT vs 2-LPT

In [ ]:
nr, nc = 1, 3
fig, axes = plt.subplots(nr, nc, figsize=(nc*5, nr*4), dpi=120)
axes = iter(axes.flat)

ax = next(axes)
ax.loglog(kh, pk, color='k', lw=1.5, label='CAMB P(k)', alpha=0.5)
pk_pt1 = pk_particle(xpert_lpt1, nvox, Lbox)
ax.loglog(pk_pt1['k'], pk_pt1['power'].real,
          color='tab:red', lw=1.5, label='Particle P(k)')
ax.set_xlabel('$k\,[h/\mathrm{Mpc}]$', fontsize=10)
ax.set_ylabel(r'$P(k)\,[h^{-3}\mathrm{Mpc}^3]$', fontsize=10)
ax.set_title('Matter power spectrum (1-LPT)', loc='left', fontsize=10)
ax.legend(loc='upper right', fontsize=10, frameon=False)

ax = next(axes)
ax.loglog(kh, pk, color='k', lw=1.5, label='CAMB P(k)', alpha=0.5)
pk_pt2 = pk_particle(xpert_lpt2, nvox, Lbox)
ax.loglog(pk_pt2['k'], pk_pt2['power'].real,
          color='tab:blue', lw=1.5, label='Particle P(k)')
ax.set_xlabel('$k\,[h/\mathrm{Mpc}]$', fontsize=10)
ax.set_title('Matter power spectrum (2-LPT)', loc='left', fontsize=10)
ax.legend(loc='upper right', fontsize=10, frameon=False)

ax = next(axes)
ax.plot(pk_pt2['k'], pk_pt2['power'].real - pk_pt1['power'].real,
        color='tab:green', lw=1.5, label=r'$P_{\mathrm{2LPT}}(k) - P_{\mathrm{1LPT}}(k)$')
ax.set_xlabel('$k\,[h/\mathrm{Mpc}]$', fontsize=10)
ax.set_title('Difference between 1-LPT and 2-LPT', loc='left', fontsize=10)
ax.legend(loc='upper right', fontsize=10, frameon=False)

plt.savefig(f'output/pk_grid_N{nmesh}_L{np.max(Lbox)}_compare.png', bbox_inches='tight', dpi=300)
plt.show()